In [ ]:
# project_script.py
"""
Interpretable ML: Causal Inference in Customer Churn Prediction
Run end-to-end: preprocess -> predictive models -> causal estimation (CausalForestDML) -> analysis -> save results.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score
import joblib

# Optional (install econml if not present)
# Install econml if not already installed
try:
    import econml
except ImportError:
    !pip install econml

from econml.dml import CausalForestDML

# === Path & output setup ===
DATA_PATH = "Telco-Customer-Churn.csv"
OUT_DIR = "results"
FIG_DIR = os.path.join(OUT_DIR, "figures")
MODEL_DIR = os.path.join(OUT_DIR, "models")
REPORT_DIR = "report"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# === 1. Load and quick EDA ===
def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    print("Loaded:", df.shape)
    return df

# === 2. Preprocessing & feature engineering ===
def preprocess(df):
    # Copy
    data = df.copy()
    # Example: standard telco churn dataset columns handling
    # Convert TotalCharges to numeric if present
    if 'TotalCharges' in data.columns:
        data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
    # Map target 'Churn' to binary
    if 'Churn' in data.columns:
        data['churn'] = data['Churn'].map({'Yes':1, 'No':0})
    # Example: Create a synthetic treatment indicator if dataset doesn't have one.
    # If your dataset already includes an intervention flag (e.g., 'received_offer'), use that column.
    if 'treatment' not in data.columns:
        # Heuristic: customers with higher monthly charge were targeted in marketing (this is just an example)
        median_monthly = data['MonthlyCharges'].median() if 'MonthlyCharges' in data.columns else 50
        data['treatment'] = (data['MonthlyCharges'] > median_monthly).astype(int)  # replace with real flag if present

    # Select features: drop identifiers and original churn column
    drop_cols = ['customerID', 'Churn'] if 'customerID' in data.columns else []
    drop_cols = [c for c in drop_cols if c in data.columns]
    features = [c for c in data.columns if c not in drop_cols + ['churn', 'treatment']]

    # Identify numeric/categorical
    num_cols = data[features].select_dtypes(include=['int64','float64']).columns.tolist()
    cat_cols = data[features].select_dtypes(include=['object','category','bool']).columns.tolist()

    # Build transformer
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

    # Fit transform to dataset (we will split later, but fit to entire dataset for convenience; for strictness, fit only on train)
    X = data[features]
    X_processed = preprocessor.fit_transform(X)

    # Get transformed column names (helpful for interpretation)
    ohe_columns = []
    if cat_cols:
        ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
        ohe_features = ohe.get_feature_names_out(cat_cols)
        ohe_columns = list(ohe_features)
    processed_cols = num_cols + ohe_columns

    X_proc_df = pd.DataFrame(X_processed, columns=processed_cols, index=data.index)
    X_proc_df['treatment'] = data['treatment'].values
    X_proc_df['churn'] = data['churn'].values

    return X_proc_df, preprocessor, num_cols, cat_cols

# === 3. Diagnostic functions for overlap / propensity ===
def fit_propensity(X, treatment_col='treatment'):
    features = [c for c in X.columns if c not in ['treatment','churn']]
    X_train = X[features]
    t = X[treatment_col]
    prop_model = LogisticRegression(max_iter=1000)
    prop_model.fit(X_train, t)
    propensity = prop_model.predict_proba(X_train)[:,1]
    # Save
    joblib.dump(prop_model, os.path.join(MODEL_DIR, "propensity_model.pkl"))
    return propensity, prop_model

def check_overlap(propensity, t, bins=10):
    # Simple checks: distribution of propensity by treatment group
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    plt.hist(propensity[t==1], bins=bins, alpha=0.6, label='Treated')
    plt.hist(propensity[t==0], bins=bins, alpha=0.6, label='Control')
    plt.legend()
    plt.title("Propensity score distribution by group")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "propensity_dist.png"))
    plt.close()

# === 4. Train outcome and CATE models ===
def train_predictive_models(X):
    features = [c for c in X.columns if c not in ['treatment','churn']]
    X_train, X_test, y_train, y_test = train_test_split(X[features], X['churn'], test_size=0.2, random_state=42, stratify=X['churn'])

    # Outcome model (for predictive baseline)
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_proba = rf.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_pred_proba)
    print("Outcome model AUC:", auc)
    joblib.dump(rf, os.path.join(MODEL_DIR, "outcome_model.pkl"))
    return rf, auc

def train_causal_forest(X, preprocessor, n_estimators=100):
    # econml CausalForestDML expects
    features = [c for c in X.columns if c not in ['treatment','churn']]
    X_feats = X[features].values
    T = X['treatment'].values
    Y = X['churn'].values

    # learners for nuisance models
    from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
    # Use RandomForestRegressor for model_y as econml expects a regressor when the target is continuous
    # (even if it's technically 0/1, it predicts probabilities/continuous values)
    model_y = RandomForestRegressor(n_estimators=200, random_state=42)
    model_t = RandomForestClassifier(n_estimators=200, random_state=42)

    # Causal forest estimator
    cf = CausalForestDML(model_t=model_t, model_y=model_y,
                         n_estimators=n_estimators, random_state=42,
                         discrete_treatment=True)
    cf.fit(Y, T, X=X_feats)
    # Estimate CATE for each sample
    cate = cf.effect(X_feats)
    # Also get confidence intervals
    lb, ub = cf.effect_interval(X_feats)
    # Save estimator
    joblib.dump(cf, os.path.join(MODEL_DIR, "causal_forest.pkl"))
    return cate, lb, ub, cf, features, X_feats # Return X_feats here

# === 5. Analysis and segmentation ===
def analyze_cate(cate, X, features):
    out_df = X.copy()
    out_df['cate'] = cate
    out_df.sort_values('cate', ascending=False, inplace=True)
    out_df.to_csv(os.path.join(OUT_DIR, "cate_estimates.csv"), index=False)
    # Plot histogram
    plt.figure(figsize=(8,4))
    plt.hist(cate, bins=40)
    plt.title("CATE distribution (estimated uplift on churn probability)")
    plt.xlabel("Estimated CATE")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "cate_hist.png"))
    plt.close()
    # Segment customers
    q1 = np.percentile(cate, 90)   # top 10% responders
    q2 = np.percentile(cate, 50)
    out_df['segment'] = 'neutral'
    out_df.loc[out_df['cate'] >= q1, 'segment'] = 'high_gain'
    out_df.loc[out_df['cate'] <= np.percentile(cate, 10), 'segment'] = 'negative_effect'
    out_df.to_csv(os.path.join(OUT_DIR, "cate_segmented.csv"), index=False)
    # Mean cate by segment
    summary = out_df.groupby('segment')['cate'].agg(['count','mean','std']).reset_index()
    summary.to_csv(os.path.join(OUT_DIR, "cate_segment_summary.csv"), index=False)
    # Save a plot of mean CATE by segment
    plt.figure(figsize=(6,4))
    plt.bar(summary['segment'], summary['mean'])
    plt.title("Average CATE by segment")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "cate_by_segment.png"))
    plt.close()
    return out_df, summary

# === 6. Diagnostics & assumptions ===
def diagnostics(X, propensity):
    # Overlap check saved previously
    check_overlap(propensity, X['treatment'])
    # Covariate balance check (simple standardized mean differences)
    def smd(x_treated, x_control):
        return (np.mean(x_treated) - np.mean(x_control)) / np.sqrt((np.var(x_treated)+np.var(x_control))/2)
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ['treatment','churn']]
    smd_list = []
    for col in numeric_cols:
        s = smd(X.loc[X['treatment']==1, col].dropna(), X.loc[X['treatment']==0, col].dropna())
        smd_list.append({'feature': col, 'smd': s})
    smd_df = pd.DataFrame(smd_list)
    smd_df.to_csv(os.path.join(OUT_DIR, "smd_balance.csv"), index=False)
    return smd_df

# === 7. Generate textual report skeleton (short automated draft). Expand manually to reach 750+ words ===
def generate_report(summary, out_df):
    # Short automated draft - fill with more interpretation manually
    lines = []
    lines.append("Causal Analysis Report\n")
    lines.append("Summary of CATE estimates and segments:\n")
    lines.append(summary.to_string(index=False))
    lines.append("\n\nInterpretation and limitations:\n")
    lines.append("1. Heterogeneity: We observe subgroups with positive CATE (marketing reduces churn probability) and small subgroup with negative CATE.\n")
    lines.append("2. Assumptions: The causal estimates rely on unconfoundedness given observed covariates and overlap. Check the smd_balance.csv for covariate imbalance.\n")
    lines.append("3. Limitations: Observational dataset & synthetic treatment flag used (replace with real intervention flag if available). Possible unobserved confounding.\n")
    lines.append("\nRecommendations (draft):\n")
    lines.append("1. Target high_gain segment with personalized retention offers.\n2. Avoid applying same intervention to negative_effect group without A/B testing.\n3. Re-run models after collecting results from a randomized trial.\n")
    with open(os.path.join(REPORT_DIR, "causal_analysis.txt"), "w") as f:
        f.write("\n".join(lines))
    with open(os.path.join(REPORT_DIR, "recommendations.txt"), "w") as f:
        f.write("1. Prioritize high_gain customers for targeted campaigns.\n2. Collect randomized trial data for top segments.\n3. Validate models quarterly.\n")
    print("Reports saved to", REPORT_DIR)

# === 8. Main pipeline ===
def main():
    df = load_data(DATA_PATH)
    X_proc, preprocessor, num_cols, cat_cols = preprocess(df)
    prop, prop_model = fit_propensity(X_proc, treatment_col='treatment')
    smd_df = diagnostics(X_proc, prop)
    rf, auc = train_predictive_models(X_proc)
    cate, lb, ub, cf, features, X_feats = train_causal_forest(X_proc, preprocessor, n_estimators=200)
    out_df, summary = analyze_cate(cate, X_proc, features)
    generate_report(summary, out_df)
    print("Done. Check the results folder.")
    return X_proc, features, cate, cf, X_feats # Return necessary variables

if __name__ == "__main__":
    X_proc, features, cate, cf, X_feats = main()
# after cf is trained:
try:
    # econml's built-in feature importance if available
    importances = cf.feature_importances(X_feats)
    # if feature names available, map them:
    feat_imp_df = pd.DataFrame({'feature': features, 'importance': importances})
    feat_imp_df.sort_values('importance', ascending=False, inplace=True)
    feat_imp_df.to_csv(os.path.join(OUT_DIR, "cate_feature_importances.csv"), index=False)
except Exception as e:
    print("Could not compute cf.feature_importances():", e)
    # fallback: train a surrogate model to predict CATE from features
    from sklearn.ensemble import RandomForestRegressor
    surrogate = RandomForestRegressor(n_estimators=200, random_state=42)
    surrogate.fit(X_feats, cate)
    importances = surrogate.feature_importances_
    feat_imp_df = pd.DataFrame({'feature': features, 'importance': importances})
    feat_imp_df.sort_values('importance', ascending=False, inplace=True)
    feat_imp_df.to_csv(os.path.join(OUT_DIR, "cate_feature_importances_surrogate.csv"), index=False)
# Run SHAP for outcome model (requires shap package)
import shap
# load outcome model and use a sample of features
outcome_model = joblib.load(os.path.join(MODEL_DIR, "outcome_model.pkl"))
X_sample = X_proc[features].sample(min(1000, len(X_proc)))
explainer = shap.Explainer(outcome_model, X_sample)
shap_vals = explainer(X_sample)
# save a plot
plt.figure()
shap.summary_plot(shap_vals, X_sample, show=False)
plt.savefig(os.path.join(FIG_DIR, "shap_outcome_summary.png"), bbox_inches='tight')
plt.close()
# Placebo test: randomize treatment and re-run a small causal forest (fast)
np.random.seed(42)
T_random = np.random.binomial(1, X_proc['treatment'].mean(), size=len(X_proc))
# Need model_y and model_t definitions from train_causal_forest, so defining them here for global scope
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
model_y = RandomForestRegressor(n_estimators=200, random_state=42)
model_t = RandomForestClassifier(n_estimators=200, random_state=42)

cf_placebo = CausalForestDML(model_t=model_t, model_y=model_y, n_estimators=100, random_state=123, discrete_treatment=True)
cf_placebo.fit(X_proc['churn'].values, T_random, X=X_feats)
placebo_effect = cf_placebo.effect(X_feats)
# Save placebo distribution plot
plt.figure(figsize=(6,3))
plt.hist(placebo_effect, bins=30)
plt.title("Placebo CATE distribution (randomized treatment)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "placebo_cate_hist.png"))
plt.close()
# Compare means
print("Mean real CATE:", np.mean(cate), "Mean placebo CATE:", np.mean(placebo_effect))

Loaded: (7043, 21)
Outcome model AUC: 0.8190562918184402
Reports saved to report
Done. Check the results folder.
Could not compute cf.feature_importances(): only length-1 arrays can be converted to Python scalars


100%|===================| 1996/2000 [05:07<00:00]       

Mean real CATE: -0.09842056485371069 Mean placebo CATE: 0.004675900455825779


<Figure size 640x480 with 0 Axes>